In [2]:
import joblib
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath('..'))

# Cargar artefacto del Sprint 2
artifact = joblib.load('../models/preprocessor.pkl')
feature_builder = artifact['feature_builder']
preprocessor    = artifact['preprocessor']

# Cargar datos
X_train = pd.read_csv('../data/processed/X_train_bal.csv')
y_train = pd.read_csv('../data/processed/y_train_bal.csv').values.ravel()
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_test  = pd.read_csv('../data/processed/y_test.csv').values.ravel()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("preprocessor.pkl cargado OK ✅")

X_train shape: (100232, 47)
X_test shape: (17275, 47)
preprocessor.pkl cargado OK ✅


In [3]:
import joblib
import pandas as pd

# Cargar datos
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

# Verificar cada modelo
modelos = ['lr', 'dt', 'knn', 'svm', 'gb']
for nombre in modelos:
    pipe = joblib.load(f'../models/baseline_{nombre}.pkl')
    pred = pipe.predict(X_test)
    print(f"baseline_{nombre}.pkl → OK ✅, predicciones: {pred[:3]}")

baseline_lr.pkl → OK ✅, predicciones: [0 0 1]
baseline_dt.pkl → OK ✅, predicciones: [1 0 0]
baseline_knn.pkl → OK ✅, predicciones: [1 1 1]
baseline_svm.pkl → OK ✅, predicciones: [0 0 0]
baseline_gb.pkl → OK ✅, predicciones: [0 0 1]


In [ ]:
import pandas as pd
import os

experiments = pd.DataFrame({
    'model':   ['gb',          'dt',          'knn',         'lr',          'svm'],
    'params':  [
        'random_state=42',
        'random_state=42',
        'default',
        'random_state=42, max_iter=1000',
        'probability=True, random_state=42'
    ],
    'date': ['2026-04-24'] * 5,

    # CV (para comparación de modelos)
    'cv_accuracy':  [0.791933, 0.787802, 0.778783, 0.734476, None],
    'cv_precision': [0.794422, 0.785204, 0.763624, 0.717195, None],
    'cv_recall':    [0.787772, 0.792461, 0.807587, 0.774284, None],
    'cv_f1':        [0.791068, 0.788799, 0.784975, 0.744639, None],
    'cv_roc_auc':   [0.877783, 0.788192, 0.853985, 0.818114, None],

    # Test (desempeño final reportado)
    'test_recall':   [0.665824, 0.584703, 0.622419, 0.651285, 0.595027],
    'test_f1':       [0.606817, 0.548690, 0.547341, 0.532426, 0.565592],
    'test_accuracy': [0.762952, 0.735745, 0.717164, 0.685731, 0.748886],

    'selected': [True, False, True, False, False],
    'notes': [
        'Mejor AUC y F1 en CV. Seleccionado para Sprint 4',
        'Buen recall en CV pero descartado frente a GB y KNN',
        'Mayor recall en CV (0.807). Seleccionado para Sprint 4',
        'Menor desempeño general. Descartado',
        'Excluido de CV por costo computacional.'
    ]
})

os.makedirs('../models', exist_ok=True)
experiments.to_csv('../models/experiments_log.csv', index=False)
print("✅ experiments_log.csv creado!")
print(experiments.to_string())

✅ experiments_log.csv creado!
  model                             params        date  cv_accuracy  cv_precision  cv_recall     cv_f1  cv_roc_auc  test_recall   test_f1  test_accuracy  selected                                                                       notes
0    gb                    random_state=42  2026-04-24     0.791933      0.794422   0.787772  0.791068    0.877783     0.665824  0.606817       0.762952      True                            Mejor AUC y F1 en CV. Seleccionado para Sprint 4
1    dt                    random_state=42  2026-04-24     0.787802      0.785204   0.792461  0.788799    0.788192     0.584703  0.548690       0.735745     False                         Buen recall en CV pero descartado frente a GB y KNN
2   knn                            default  2026-04-24     0.778783      0.763624   0.807587  0.784975    0.853985     0.622419  0.547341       0.717164      True                      Mayor recall en CV (0.807). Seleccionado para Sprint 4
3    lr     ra

## Flujo completo: de datos crudos a predicción

1. **Datos crudos** → `data/raw/01-hotel_bookings.csv`
2. **Limpieza** → `05_data_cleaning.ipynb` → `data/interim/`
3. **Feature Engineering** → `06_feature_eng.ipynb`
4. **Balanceo de clases** → `07_class_balance.ipynb`
5. **Pipeline integrado** → `08_pipeline.ipynb` 
   → genera `models/preprocessor.pkl`
   → genera `data/processed/X_train_bal.csv`, `X_test.csv`, etc.
6. **Entrenamiento baseline** → `09_baseline_models.ipynb`
   → genera `models/baseline_*.pkl`
7. **Evaluación** → `10_evaluation.ipynb`
8. **Predicción final** → cargar `baseline_XX.pkl` → `.predict(X_test)`

In [9]:
import os

# Estructura para Sprint 4
os.makedirs('../src', exist_ok=True)

# Archivo vacío que usará el tuner en Sprint 4
with open('../src/tuning.py', 'w') as f:
    f.write("""# Sprint 4: Optimización de hiperparámetros
# Este módulo contendrá las funciones de tuning
# para los modelos seleccionados: GB, KNN, SVM

# Modelos candidatos para tuning:
# - baseline_gb.pkl - GradientBoosting
# - baseline_knn.pkl - KNeighbors  
# - baseline_svm.pkl - SVM

def tune_model(pipe, param_grid, X, y, cv):
    pass  # TODO Sprint 4
""")
print("✅ src/tuning.py creado para Sprint 4")

✅ src/tuning.py creado para Sprint 4
